# NB_B1 · Silver · Réseau piéton OSM

**Entrée (bronze)** : extraits Geofabrik `.osm.pbf` par région, dans `Files/bronze/osm/geofabrik/{region}/{date}/`.

**Sorties (schéma `silver` du lakehouse `lh_mobility_citiz`, tables Delta)** :
| Table | Une ligne = | À quoi ça sert |
|---|---|---|
| `reseau_pieton_troncons` | un petit bout de chemin entre deux points, avec sa longueur | les « routes » du GPS piéton (gold : zones à 5 min à pied) |
| `reseau_pieton_points` | un point de jonction, avec son nombre de chemins | les « intersections » : impasse, point de passage ou carrefour |
| `reseau_pieton_suivi` | une région traitée | le journal : date des données, volumes, durée |

**Étapes** : garder les voies praticables à pied → les découper en tronçons mesurés → supprimer les doublons entre régions voisines → repérer impasses et carrefours → contrôles qualité.

⚠️ `lh_mobility_citiz` doit être le lakehouse **par défaut** de ce notebook.

In [1]:
%pip install osmium --quiet

## 1. Paramètres
Tout est dans le lakehouse `lh_mobility_citiz` (lakehouse **par défaut** de ce notebook) : on lit `Files/bronze`, on écrit dans `Files/silver` et dans le schéma de tables `silver`.

In [2]:
BRONZE_OSM = "/lakehouse/default/Files/bronze/osm/geofabrik"
STAGING    = "/lakehouse/default/Files/silver/osm/_staging/reseau_pieton_troncons"   # Parquet intermédiaire, par région
SCHEMA     = "silver"                                                        # schéma des tables Delta
LOCAL      = "/tmp/osm_silver"                                               # disque local du driver

REGIONS = None        # None = toutes les régions présentes en bronze ; sinon ex. ["alsace", "lorraine"]
FORCE   = False       # False = on saute les régions déjà traitées (reprise après coupure)

## 2. Règles de filtrage et découpage en tronçons
Une voie est gardée si elle a un tag `highway` praticable à pied (règles inspirées du filtre `walk` d'OSMnx) :
- exclues : autoroutes, chantiers, projets, voies abandonnées, quais, `area=yes` ;
- exclues si `foot=no`, `access=private` (sauf `foot=yes`), `service=private` ;
- `trunk` et `cycleway` gardées **seulement** si `foot` les autorise explicitement.

Chaque voie est découpée en tronçons « point de départ → point d'arrivée », avec leur longueur (formule de haversine).
Les valeurs OSM, en anglais, sont traduites en libellés français (ex. `footway` → « Chemin piéton »).

In [3]:
import math
import osmium
import pyarrow as pa
import pyarrow.parquet as pq

# ---------------------------------------------------------------------------
# Règles « praticable à pied » (inspirées du filtre "walk" d'OSMnx, documenté)
# ---------------------------------------------------------------------------
HIGHWAY_EXCLUS = {
    "motorway", "motorway_link", "trunk_link", "bus_guideway", "busway", "raceway",
    "construction", "proposed", "planned", "abandoned", "razed", "disused",
    "platform", "escape", "corridor", "elevator", "no",
}
FOOT_OUI = {"yes", "designated", "permissive", "official"}
FOOT_NON = {"no", "private", "use_sidepath"}
ACCES_NON = {"no", "private"}


def est_pieton(tags) -> bool:
    """True si la voie OSM est praticable à pied."""
    hw = tags.get("highway")
    if hw is None or hw in HIGHWAY_EXCLUS:
        return False
    if tags.get("area") == "yes":          # places en polygone : pas des tronçons linéaires
        return False
    foot = tags.get("foot")
    if foot in FOOT_NON:                   # interdit explicitement aux piétons
        return False
    if tags.get("access") in ACCES_NON and foot not in FOOT_OUI:
        return False                       # voie privée, sauf exception piétonne
    if tags.get("service") == "private":
        return False
    if hw in ("trunk", "cycleway") and foot not in FOOT_OUI:
        return False                       # voies rapides et pistes cyclables : seulement si foot autorisé
    return True


# ---------------------------------------------------------------------------
# Traductions des valeurs OSM (anglais) en libellés compréhensibles
# ---------------------------------------------------------------------------
TYPE_VOIE_FR = {
    "footway": "Chemin piéton", "pedestrian": "Rue piétonne", "path": "Sentier",
    "steps": "Escalier", "living_street": "Zone de rencontre", "residential": "Rue résidentielle",
    "service": "Voie de desserte", "track": "Chemin agricole ou forestier",
    "cycleway": "Piste cyclable ouverte aux piétons", "bridleway": "Chemin équestre",
    "unclassified": "Petite route", "road": "Voie non qualifiée",
    "tertiary": "Route locale", "tertiary_link": "Route locale",
    "secondary": "Route secondaire", "secondary_link": "Route secondaire",
    "primary": "Route principale", "primary_link": "Route principale",
    "trunk": "Voie rapide ouverte aux piétons",
}
PRECISION_PIETON_FR = {"sidewalk": "Trottoir", "crossing": "Passage piéton",
                       "access_aisle": "Allée d'accès", "traffic_island": "Îlot"}
REVETEMENT_FR = {
    "asphalt": "Enrobé (bitume)", "paved": "Revêtu", "concrete": "Béton",
    "paving_stones": "Pavés", "sett": "Pavés", "cobblestone": "Pavés",
    "gravel": "Gravier", "fine_gravel": "Gravillons", "compacted": "Terre compactée",
    "unpaved": "Non revêtu", "ground": "Terre", "dirt": "Terre", "earth": "Terre",
    "grass": "Herbe", "sand": "Sable", "wood": "Bois", "metal": "Métal",
}
OUI_NON_FR = {"yes": "oui", "no": "non", "limited": "partiel", "designated": "oui"}


def traduire(valeur, dico):
    """Traduit une valeur OSM ; garde la valeur d'origine si elle n'est pas dans le dictionnaire."""
    return None if valeur is None else dico.get(valeur, valeur)


def haversine_m(lon1, lat1, lon2, lat2) -> float:
    """Distance à vol d'oiseau en mètres entre deux points (lon/lat WGS84)."""
    r = 6_371_008.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


# Colonnes de la table reseau_pieton_troncons
SCHEMA_TRONCONS = pa.schema([
    ("id_voie_osm", pa.int64()),          # identifiant de la rue/chemin dans OpenStreetMap
    ("numero_troncon", pa.int32()),       # position du tronçon dans la voie (0, 1, 2…)
    ("id_point_depart", pa.int64()),
    ("id_point_arrivee", pa.int64()),
    ("depart_longitude", pa.float64()), ("depart_latitude", pa.float64()),
    ("arrivee_longitude", pa.float64()), ("arrivee_latitude", pa.float64()),
    ("longueur_m", pa.float64()),
    ("type_voie", pa.string()),           # libellé en français
    ("type_voie_osm", pa.string()),       # valeur OSM d'origine (traçabilité)
    ("nom_voie", pa.string()),
    ("precision_pieton", pa.string()),    # trottoir, passage piéton…
    ("revetement", pa.string()),
    ("eclairage", pa.string()),
    ("accessible_fauteuil", pa.string()),
    ("pente", pa.string()),
    ("region_source", pa.string()),       # fichier régional Geofabrik d'origine
])


class WalkHandler(osmium.SimpleHandler):
    """Découpe chaque voie piétonne en tronçons (paires de points consécutifs)
    et les écrit en Parquet par blocs, pour ne pas saturer la mémoire."""

    def __init__(self, region: str, out_path: str, chunk_size: int = 1_000_000):
        super().__init__()
        self.region = region
        self.chunk_size = chunk_size
        self.writer = pq.ParquetWriter(out_path, SCHEMA_TRONCONS, compression="zstd")
        self._reset()
        self.nb_voies_lues = self.nb_voies_pietonnes = self.nb_troncons = self.nb_troncons_ignores = 0

    def _reset(self):
        self.cols = {f.name: [] for f in SCHEMA_TRONCONS}

    def _flush(self):
        if self.cols["id_voie_osm"]:
            self.writer.write_table(pa.table(self.cols, schema=SCHEMA_TRONCONS))
            self._reset()

    def way(self, w):
        if "highway" not in w.tags:
            return
        self.nb_voies_lues += 1
        if not est_pieton(w.tags):
            return
        self.nb_voies_pietonnes += 1
        t = w.tags
        hw = t.get("highway")
        attributs = {
            "type_voie": TYPE_VOIE_FR.get(hw, "Autre"),
            "type_voie_osm": hw,
            "nom_voie": t.get("name"),
            "precision_pieton": traduire(t.get("footway"), PRECISION_PIETON_FR),
            "revetement": traduire(t.get("surface"), REVETEMENT_FR),
            "eclairage": traduire(t.get("lit"), OUI_NON_FR),
            "accessible_fauteuil": traduire(t.get("wheelchair"), OUI_NON_FR),
            "pente": t.get("incline"),
            "region_source": self.region,
        }
        points, c = w.nodes, self.cols
        for i in range(len(points) - 1):
            a, b = points[i], points[i + 1]
            if not (a.location.valid() and b.location.valid()):
                self.nb_troncons_ignores += 1      # point absent de l'extrait régional
                continue
            alon, alat, blon, blat = a.location.lon, a.location.lat, b.location.lon, b.location.lat
            c["id_voie_osm"].append(w.id); c["numero_troncon"].append(i)
            c["id_point_depart"].append(a.ref); c["id_point_arrivee"].append(b.ref)
            c["depart_longitude"].append(alon); c["depart_latitude"].append(alat)
            c["arrivee_longitude"].append(blon); c["arrivee_latitude"].append(blat)
            c["longueur_m"].append(haversine_m(alon, alat, blon, blat))
            for k, v in attributs.items():
                c[k].append(v)
            self.nb_troncons += 1
        if len(c["id_voie_osm"]) >= self.chunk_size:
            self._flush()

    def close(self):
        self._flush()
        self.writer.close()


def extraire_troncons(pbf_path: str, region: str, out_path: str) -> dict:
    """Lit un .osm.pbf et écrit les tronçons piétons dans out_path (Parquet)."""
    h = WalkHandler(region, out_path)
    try:
        # locations=True : libosmium (C++) garde les coordonnées des points en mémoire
        # et les injecte dans les voies ; les points ne passent jamais par Python.
        h.apply_file(pbf_path, locations=True, idx="flex_mem")
    finally:
        h.close()
    return {"region": region, "nb_voies_lues": h.nb_voies_lues,
            "nb_voies_pietonnes": h.nb_voies_pietonnes, "nb_troncons": h.nb_troncons,
            "nb_troncons_ignores": h.nb_troncons_ignores}


## 3. Traitement région par région (bronze → fichiers intermédiaires)

In [4]:
import os, json, time, shutil
import pandas as pd
from datetime import datetime, timezone

os.makedirs(STAGING, exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)

def sous_dossiers(path):
    return sorted(d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d)))

regions = REGIONS or sous_dossiers(BRONZE_OSM)
print(f"{len(regions)} région(s) :", regions)

runs = []
for region in regions:
    out_final = f"{STAGING}/{region}.parquet"
    if os.path.exists(out_final) and not FORCE:
        print(f"⏭  {region} déjà traitée")
        continue

    # Dernière date disponible en bronze pour cette région
    date = sous_dossiers(f"{BRONZE_OSM}/{region}")[-1]
    src_dir = f"{BRONZE_OSM}/{region}/{date}"
    pbf = next(f for f in os.listdir(src_dir) if f.endswith(".osm.pbf"))
    manifeste = {}
    if os.path.exists(f"{src_dir}/_manifest.json"):
        with open(f"{src_dir}/_manifest.json") as fh:
            manifeste = json.load(fh)

    t0 = time.time()
    local_pbf, local_out = f"{LOCAL}/{region}.osm.pbf", f"{LOCAL}/{region}.parquet"
    shutil.copy(f"{src_dir}/{pbf}", local_pbf)       # copie sur disque local : lecture osmium plus fiable
    stats = extraire_troncons(local_pbf, region, local_out)
    shutil.copy(local_out, out_final)                 # dépôt dans Files/silver/osm/_staging
    os.remove(local_pbf); os.remove(local_out)

    stats.update(date_donnees_bronze=date, fichier_source=pbf,
                 taille_fichier_mo=round(os.path.getsize(f"{src_dir}/{pbf}") / 1e6, 1),
                 empreinte_md5=str(manifeste.get("md5") or manifeste.get("md5_local") or ""),
                 duree_traitement_s=round(time.time() - t0, 1),
                 date_traitement=datetime.now(timezone.utc).isoformat())
    runs.append(stats)
    print(f"✅ {region:30s} {stats['nb_voies_pietonnes']:>9,} voies  {stats['nb_troncons']:>11,} tronçons  {stats['duree_traitement_s']:>7} s")

if runs:
    (spark.createDataFrame(pd.DataFrame(runs)).write.mode("append")
          .option("mergeSchema", "true").saveAsTable(f"{SCHEMA}.reseau_pieton_suivi"))

## 4. Dédoublonnage et écriture des tables Delta
Les extraits Geofabrik se chevauchent : une voie qui traverse une frontière régionale est présente **en entier** dans les deux fichiers.
On garde une seule occurrence par couple `(id_voie_osm, numero_troncon)`, celle de la première région par ordre alphabétique, pour que le résultat soit reproductible.

In [5]:
from pyspark.sql import functions as F, Window

tous = spark.read.parquet(STAGING.replace("/lakehouse/default/", ""))

# 1) Tronçons sans doublons
w = Window.partitionBy("id_voie_osm", "numero_troncon").orderBy("region_source")
troncons = (tous.withColumn("rang", F.row_number().over(w))
                .filter("rang = 1").drop("rang"))
(troncons.write.mode("overwrite").option("overwriteSchema", "true")
         .format("delta").saveAsTable(f"{SCHEMA}.reseau_pieton_troncons"))

# 2) Points : chaque extrémité de tronçon, avec le nombre de chemins qui en partent
troncons = spark.table(f"{SCHEMA}.reseau_pieton_troncons")
extremites = (
    troncons.select(F.col("id_point_depart").alias("id_point"),
                    F.col("depart_longitude").alias("longitude"), F.col("depart_latitude").alias("latitude"),
                    F.col("id_point_arrivee").alias("voisin"))
    .unionByName(
    troncons.select(F.col("id_point_arrivee").alias("id_point"),
                    F.col("arrivee_longitude").alias("longitude"), F.col("arrivee_latitude").alias("latitude"),
                    F.col("id_point_depart").alias("voisin"))))
points = (extremites.groupBy("id_point")
          .agg(F.min("longitude").alias("longitude"), F.min("latitude").alias("latitude"),
               F.countDistinct("voisin").alias("nb_chemins"))
          .withColumn("type_point",
                      F.when(F.col("nb_chemins") == 1, "Impasse")
                       .when(F.col("nb_chemins") == 2, "Point de passage")
                       .otherwise("Carrefour")))
(points.write.mode("overwrite").option("overwriteSchema", "true")
       .format("delta").saveAsTable(f"{SCHEMA}.reseau_pieton_points"))
print(f"Tables écrites : {SCHEMA}.reseau_pieton_troncons, {SCHEMA}.reseau_pieton_points")

## 5. Contrôles qualité

In [6]:
avant = spark.read.parquet(STAGING.replace("/lakehouse/default/", "")).count()
t     = spark.table(f"{SCHEMA}.reseau_pieton_troncons")
p     = spark.table(f"{SCHEMA}.reseau_pieton_points")
nb_t, nb_p = t.count(), p.count()
uniques = t.select("id_voie_osm", "numero_troncon").distinct().count()
km    = t.agg(F.sum("longueur_m")).first()[0] / 1000

print(f"Tronçons avant dédoublonnage : {avant:,}")
print(f"Tronçons après dédoublonnage : {nb_t:,}  (doublons retirés : {avant - nb_t:,})")
print(f"Contrôle des doublons        : {'OK' if uniques == nb_t else 'ERREUR : il reste des doublons'}")
print(f"Points de jonction           : {nb_p:,}")
print(f"Longueur totale du réseau    : {km:,.0f} km")
print(f"Coordonnées manquantes       : {t.filter('depart_longitude IS NULL OR arrivee_longitude IS NULL').count()}")

# Kilomètres par type de voie
display(t.groupBy("type_voie").agg(F.count("*").alias("nb_troncons"), F.round(F.sum("longueur_m") / 1000).alias("km"))
         .orderBy(F.desc("km")))
# Kilomètres par région source
display(t.groupBy("region_source").agg(F.count("*").alias("nb_troncons"), F.round(F.sum("longueur_m") / 1000).alias("km"))
         .orderBy("region_source"))
# Impasses, points de passage, carrefours
display(p.groupBy("type_point").count().orderBy("type_point"))
# Longueur des tronçons : médiane, 95 % et 99,9 % des tronçons font moins de…
display(t.select(F.percentile_approx("longueur_m", [0.5, 0.95, 0.999]).alias("longueur_m_mediane_95pct_999pct"),
                 F.round(F.max("longueur_m")).alias("longueur_m_max")))

## Limites connues (à assumer en soutenance)
- **Complétude OSM inégale** : trottoirs détaillés en ville, souvent absents à la campagne, où l'on marche donc le long des routes, qui restent incluses.
- **Places piétonnes** (`area=yes`) exclues : la traversée d'une grande place est un peu sous-estimée.
- **Pas de relief** : longueur à plat ; les escaliers sont gardés et repérables via `type_voie` = « Escalier ».
- **Anciennes régions** : `region_source` suit le découpage Geofabrik ; le rattachement aux régions et départements actuels se fera par jointure spatiale avec les contours IGN ou la grille INSEE.
- **Licence** : données © contributeurs OpenStreetMap, ODbL, à citer dans la liste des sources.